# Build a Multi-Output GP in GPJax

## Imports

In [1]:
# %load ~/dev/marthaler/header.py
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

In [5]:
# Enable Float64 for more stable matrix inversions.
import jax

import jax.numpy as jnp
import numpy as np
import jax.random as jr
from jaxtyping import Array, ArrayLike, Bool, Float, Int
from typing import Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

## Define MultiOutput Kernel(s)

In [6]:
import abc
from gpjax.kernels.base import AbstractKernel, CombinationKernel

In [7]:
class MultioutputKernel(AbstractKernel):
    """
    Multi Output Kernel class.

    This kernel can represent correlation between outputs of different datapoints.

    The `full_output_cov` argument holds whether the kernel should calculate
    the covariance between the outputs. In case there is no correlation but
    `full_output_cov` is set to True the covariance matrix will be filled with zeros
    until the appropriate size is reached.
    """

    @property
    @abc.abstractmethod
    def num_latent_gps(self) -> Int:
        """The number of latent GPs in the multioutput kernel"""
        raise NotImplementedError

    @property
    @abc.abstractmethod
    def latent_kernels(self) -> Tuple[AbstractKernel, ...]:
        """The underlying kernels in the multioutput kernel"""
        raise NotImplementedError

    @abc.abstractmethod
    # @check_shapes(
    #     "X: [batch..., N, D]",
    #     "X2: [batch2..., N2, D]",
    #     "return: [batch..., N, P, batch2..., N2, P] if full_output_cov and (X2 is not None)",
    #     "return: [P, batch..., N, batch2..., N2] if not full_output_cov and (X2 is not None)",
    #     "return: [batch..., N, P, N, P] if full_output_cov and (X2 is None)",
    #     "return: [P, batch..., N, N] if not full_output_cov and (X2 is None)",
    # )
    def K(
        self,
        X: Float[Array, "batch ... N D"],
        X2: Float[Array, "batch2 ... N2 D"] | None = None,
        full_output_cov: Bool = True,
    ) -> Array:
        """
        Returns the correlation of f(X) and f(X2), where f(.) can be multi-dimensional.

        :param X: data matrix
        :param X2: data matrix
        :param full_output_cov: calculate correlation between outputs.
        :return: cov[f(X), f(X2)]
        """
        raise NotImplementedError

    @abc.abstractmethod
    # @check_shapes(
    #     "X: [batch..., N, D]",
    #     "return: [batch..., N, P, P] if full_output_cov",
    #     "return: [batch..., N, P] if not full_output_cov",
    # )
    def K_diag(
        self,
        X: Float[Array, "batch ... N D"],
        full_output_cov: Bool = True,
    ) -> Array:
        """
        Returns the correlation of f(X) and f(X), where f(.) can be multi-dimensional.

        :param X: data matrix
        :param full_output_cov: calculate correlation between outputs.
        :return: var[f(X)]
        """
        raise NotImplementedError

    # @check_shapes(
    #     "X: [batch..., N, D]",
    #     "X2: [batch2..., N2, D]",
    #     "return: [batch..., N, P, batch2..., N2, P] if full_cov and full_output_cov and (X2 is not None)",
    #     "return: [P, batch..., N, batch2..., N2] if full_cov and (not full_output_cov) and (X2 is not None)",
    #     "return: [batch..., N, P, N, P] if full_cov and full_output_cov and (X2 is None)",
    #     "return: [P, batch..., N, N] if full_cov and (not full_output_cov) and (X2 is None)",
    #     "return: [batch..., N, P, P] if (not full_cov) and full_output_cov and (X2 is None)",
    #     "return: [batch..., N, P] if (not full_cov) and (not full_output_cov) and (X2 is None)",
    # )
    def __call__(
        self,
        X: Float[Array, "batch ... N D"],
        X2: Float[Array, "batch2 ... N2 D"] | None = None,
        *,
        full_cov: Bool = False,
        full_output_cov: Bool = True,
        presliced: Bool = False,
    ) -> Array:
        # if not presliced:
        #     X, X2 = self.slice(X, X2)
        if not full_cov and X2 is not None:
            raise ValueError(
                "Ambiguous inputs: passing in `X2` is not compatible with `full_cov=False`."
            )
        if not full_cov:
            return self.K_diag(X, full_output_cov=full_output_cov)
        return self.K(X, X2, full_output_cov=full_output_cov)

In [49]:
class SeparateIndependent():
    """
    - Separate: we use different kernel for each output latent
    - Independent: Latents are uncorrelated a priori.
    """

    def __init__(
        self, kernels: Sequence[AbstractKernel],
    ) -> None:
        self.kernels=kernels

    @property
    def num_latent_gps(self) -> Int:
        return len(self.kernels)

    @property
    def latent_kernels(self) -> Tuple[AbstractKernel, ...]:
        """The underlying kernels in the multioutput kernel"""
        return tuple(self.kernels)

    def K(
        self,
        X: Float[Array, "batch ... N D"],
        X2: Float[Array, "batch2 ... N2 D"] | None = None,
        full_output_cov: Bool = True,
    ) -> Array:
        #rank = jnp.linalg.matrix_rank(X) - 1
        #rank = X.shape[1]
        if X2 is None:
            # if full_output_cov:
            #     Kxxs = jnp.stack([k.K(X, X2) for k in self.kernels], axis=-1)
            #     perm = jnp.concat(
            #         [
            #             jnp.arange(rank),
            #             [rank + 1, rank, rank + 2],
            #         ],
            #         0,
            #     )
            #     return jnp.transpose(jnp.linalg.diag(Kxxs), perm)#, "[batch..., N, P, N, P]")
            # else:
            jnp.stack([k.K(X, X2) for k in self.kernels], axis=0)

        else:
            #rank2 = X2.shape[1]
            #rank2 = jnp.linalg.matrix_rank(X2) - 1
            # if full_output_cov:
            #     Kxxs = jnp.stack([k.K(X, X2) for k in self.kernels], axis=-1)#,"[batch..., N, batch2..., N2, P]",
            #     perm = jnp.concat(
            #         [
            #             jnp.arange(rank),
            #             [rank + rank2],
            #             rank + tf.range(rank2),
            #             [rank + rank2 + 1],
            #         ],
            #         0,
            #     )
            #     return jnp.transpose(jnp.linalg.diag(Kxxs), perm)#, "[batch..., N, P, batch2..., N2, P]"
            # else:
            jnp.stack([k.K(X, X2) for k in self.kernels], axis=0)#,"[P, batch..., N, batch2..., N2]",

    def K_diag(
        self,
        X: Float[Array, "batch ... N D"],
        full_output_cov: Bool = True,
    ) -> Array:
        stacked = jnp.stack([k.K_diag(X) for k in self.kernels], axis=-1)#, "[batch..., N, P]")
        if full_output_cov:
            return jnp.linalg.diag(stacked)#, "[batch..., N, P, P]")
        else:
            return stacked

    def __call__(
        self,
        X: Float[Array, "batch ... N D"],
        X2: Float[Array, "batch2 ... N2 D"] | None = None,
        *,
        full_cov: Bool = False,
        full_output_cov: Bool = True,
        presliced: Bool = False,
    ) -> Array:
        # if not presliced:
        #     X, X2 = self.slice(X, X2)
        # if not full_cov and X2 is not None:
        #     raise ValueError(
        #         "Ambiguous inputs: passing in `X2` is not compatible with `full_cov=False`."
        #     )
        # if not full_cov:
        #     return self.K_diag(X, full_output_cov=full_output_cov)
        return self.K(X, X2, full_output_cov=full_output_cov)

## Generate Data

In [20]:
num_points = 100  # number of points
input_dim = 1  # number of input dimensions
num_inducing_points = 15  # number of inducing points
num_gps = 2  # number of latent GPs
output_dim = 3  # number of observations = output dimensions

In [21]:
key = jr.key(42)

In [22]:
def generate_data(key: ArrayLike, N: Int) -> Tuple[Array, Array]:
    key, subkey = jr.split(key)
    X = jr.uniform(subkey, shape=(N,1)) * 10 - 5  # Inputs = num_points x input_dim
    G = jnp.hstack((0.5 * np.sin(3 * X) + X, 3.0 * np.cos(X) - X))  # G = num_points x num_gps
    W = jnp.array([[0.5, -0.3, 1.5], [-0.4, 0.43, 0.0]])  # num_gps x output_dim
    F = jnp.matmul(G, W)  # num_points x output_dim
    key, subkey = jr.split(key)
    Y = F + 0.2 * jr.normal(subkey,shape=F.shape)

    return X, Y

In [23]:
X,y = generate_data(key, num_points)
X.shape, y.shape

((100, 1), (100, 3))

In [24]:
# Try it with no X2
import gpjax

In [25]:
kernels = [gpjax.kernels.RBF()]*num_gps

In [50]:
mo_kernel = SeparateIndependent(kernels)

In [ ]:
def initialise_gp(kernel, mean, dataset):
    prior = gpx.gps.Prior(mean_function=mean, kernel=kernel)
    likelihood = gpx.likelihoods.Gaussian(
        num_datapoints=dataset.n, obs_stddev=jnp.array([1.0e-3], dtype=jnp.float64)
    )
    posterior = prior * likelihood
    return posterior


# Define the velocity GP
mean = gpx.mean_functions.Zero()
kernel = VelocityKernel()
velocity_posterior = initialise_gp(kernel, mean, dataset_train)